# Merging and Stratifying across datasets and labels

Dataset naming schema:  "dataset_label_type.parquet"

where:
- dataset is the name of the dataset
- label is true/mixed/false
- type stands for how the data is constructed:
    1. no type: has article text
    2. claimonly: missing article text
    3. filtered: unrelated article text to claim, needs to be set to null.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
from pathlib import Path

BASE_DIR = Path.cwd().resolve()                 # for notebooks (.ipynb)

# If BASE_DIR is .../preprocess_scripts, then PROJECT_ROOT is its parent
PROJECT_ROOT = BASE_DIR.parent                      # .../WAI

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("BASE_DIR:", BASE_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

datasets = ['coaid','fakehealth','fakenewsnet','hover','liar']
labels = ['true', 'mixed','false']
types = ['','claimonly','filtered']

EXT = '.parquet'

def build_filename(dataset: str, label: str, t: str) -> Path:
    """
    Build path like:
      coaid_true.parquet
      coaid_true_claimonly.parquet
      coaid_true_filtered.parquet
    under ../data/processed
    """
    type_suffix = f"_{t}" if t else ""
    fname = f"{dataset}_{label}{type_suffix}{EXT}"
    return PROCESSED_DIR / dataset /fname

BASE_DIR: /Users/ethanbobrik/Projects/WAI/preprocess_scripts
PROJECT_ROOT: /Users/ethanbobrik/Projects/WAI
DATA_DIR: /Users/ethanbobrik/Projects/WAI/data
PROCESSED_DIR: /Users/ethanbobrik/Projects/WAI/data/processed


In [2]:
all_dfs = []

for ds in datasets:
    for lab in labels:
        for t in types:
            path = build_filename(ds, lab, t)
            if not path.exists():
                # Skip non-existent combinations
                continue

            if EXT == ".parquet":
                df = pd.read_parquet(path)
            else:
                df = pd.read_csv(path)

            # Sanity check: required columns from your schema
            required_cols = [
                "id", "label", "claim_text", "dataset", "article_text",
                "label_raw", "label_3way", "label_mode", "content_status",
                "label_confidence", "label_bin", "source_id",
                "claim_norm_hash", "lang", "content_char_len"
            ]
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                raise ValueError(f"Missing columns {missing} in file {path}")

            # For filtered type: force article_text to null before concatenation
            if t == "filtered":
                df["article_text"] = None
                df["content_status"] = "title_only"

            all_dfs.append(df)

if not all_dfs:
    raise ValueError("No dataframes were loaded. Check file paths/patterns and EXT.")

full_df = pd.concat(all_dfs, ignore_index=True)

In [4]:
if "claim_norm_hash" in full_df.columns:
    full_df = full_df.drop_duplicates(subset=["claim_norm_hash"]).reset_index(drop=True)

if "label" not in full_df.columns:
    raise ValueError("Expected a 'label' column in the unified dataframe.")

if "dataset" not in full_df.columns:
    raise ValueError("Expected a 'dataset' column in the unified dataframe.")

full_df['strat_key'] = full_df['dataset'].astype(str) + '_' + full_df['label'].astype(str)
full_df['label_confidence'] = full_df['label_confidence'].map({1.0: 'gold'})

trainval_df, test_df = train_test_split(
    full_df,
    test_size=0.1,
    random_state=42,
    stratify=full_df['strat_key']
)

train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.22222,
    random_state=42,
    stratify=trainval_df['strat_key']
)

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

unified_df = pd.concat([train_df,val_df,test_df])

In [5]:
print("Total rows:", len(full_df))
print(unified_df["split"].value_counts(), "\n")

# 1) Rows per split × dataset
print("=== Count by split and dataset ===")
print(
    unified_df
    .groupby(["split", "dataset"])
    .size()
    .unstack("dataset")
    .fillna(0)
    .astype(int)
)
print("\n")

# 2) Rows per split × label (canonical 3-way)
print("=== Count by split and label ===")
print(
    unified_df
    .groupby(["split", "label"])
    .size()
    .unstack("label")
    .fillna(0)
    .astype(int)
)
print("\n")

# 3) Rows per split × dataset × label (tiny summary)
print("=== Count by split, dataset, label (head) ===")
print(
    unified_df
    .groupby(["split", "dataset", "label"])
    .size()
    .reset_index(name="count")
    .head(30)  # increase if you want to see all
)
print("\n")

# 4) Label-mode mix: ternary vs binary per split
print("=== Label mode (0 = LIAR, 1 = binary) by split ===")
print(
    unified_df
    .groupby(["split", "label_mode"])
    .size()
    .unstack("label_mode")
    .fillna(0)
    .astype(int)
)
print("\n")

# 5) Binary label distribution (for non-LIAR rows) per split
print("=== Binary label_bin distribution by split (non-LIAR only) ===")
non_liar = unified_df[unified_df["label_mode"] == 1].copy()
if not non_liar.empty:
    print(
        non_liar
        .groupby(["split", "label_bin"])
        .size()
        .unstack("label_bin")
        .fillna(0)
        .astype(int)
    )
else:
    print("No non-LIAR rows found.")
print("\n")

# 6) Content status distribution (full_article / partial / title_only)
print("=== content_status distribution by split ===")
if "content_status" in unified_df.columns:
    print(
        unified_df
        .groupby(["split", "content_status"])
        .size()
        .unstack("content_status")
        .fillna(0)
        .astype(int)
    )
    print("\n")

# 7) Language distribution by split (just in case)
print("=== Language distribution by split ===")
if "lang" in unified_df.columns:
    print(
        unified_df
        .groupby(["split", "lang"])
        .size()
        .unstack("lang")
        .fillna(0)
        .astype(int)
    )
    print("\n")

# 8) Basic numeric stats for content_char_len per split
print("=== content_char_len summary by split ===")
if "content_char_len" in unified_df.columns:
    print(
        unified_df
        .groupby("split")["content_char_len"]
        .describe()
        .round(2)
    )
    print("\n")

# 9) Optional: per dataset + split stats for article length
print("=== content_char_len summary by split and dataset (head) ===")
if "content_char_len" in unified_df.columns:
    stats_len = (
        unified_df
        .groupby(["split","dataset"])["content_char_len"]
        .describe()
        .round(2)
        .reset_index()
    )
    print(stats_len.head(20))

Total rows: 56641
split
train    39648
val      11328
test      5665
Name: count, dtype: int64 

=== Count by split and dataset ===
dataset  coaid  fakehealth  fakenewsnet  hover  liar
split                                               
test       569         216         1004   2599  1277
train     3984        1510         7028  18191  8935
val       1138         432         2008   5197  2553


=== Count by split and label ===
label  false  mixed   true
split                     
test    1962   1186   2517
train  13735   8301  17612
val     3925   2371   5032


=== Count by split, dataset, label (head) ===
    split      dataset  label  count
0    test        coaid  false     94
1    test        coaid   true    475
2    test   fakehealth  false     72
3    test   fakehealth  mixed     71
4    test   fakehealth   true     73
5    test  fakenewsnet  false    530
6    test  fakenewsnet   true    474
7    test        hover  false    912
8    test        hover  mixed    397
9    test      

In [6]:
train_df.drop(columns=['strat_key','split'],inplace=True)
val_df.drop(columns=['strat_key','split'],inplace=True)
test_df.drop(columns=['strat_key','split'],inplace=True)

In [7]:
out_dir = PROCESSED_DIR / "final_datasets"
out_dir.mkdir(parents=True, exist_ok=True)
train_df.to_parquet(out_dir / "unified_train.parquet", index=False)
val_df.to_parquet(out_dir / "unified_val.parquet", index=False)
test_df.to_parquet(out_dir / "unified_test.parquet", index=False)